# Árbol de decisión con las 30 variables seleccionadas

Comparamos el árbol de decisión añadido al proyecto con el baseline de regresión logística usando la partición temporal completa del notebook final: entrenamiento anterior a 2015 y validación 2015–2016. La prueba desde 2017 queda reservada.

El conjunto de 30 variables es el resultado del RFE agrupado del notebook `05_seleccion_variables.ipynb`. Se comparan el baseline logístico original (65 variables), la logística con las mismas 30 variables y el árbol con esas 30 variables. Así separamos el efecto de reducir variables del efecto de cambiar de modelo.

> **Resultados históricos:** la logística base y el árbol de este notebook usan el conjunto anterior de 65 predictoras y una selección RFE derivada de ese conjunto. La exploración final actual conserva 59 predictoras con menos del 10% de faltantes; estas métricas no son comparables con el nuevo baseline.


In [1]:
from pathlib import Path
import sys, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

raiz=Path.cwd()
if raiz.name=="notebooks": raiz=raiz.parent
if str(raiz) not in sys.path: sys.path.insert(0,str(raiz))
from src.config import localizar_datos, ANIO_CORTE_VALIDACION, ANIO_CORTE_PRUEBA, UMBRAL_FALTANTES, SEMILLA
from src.data import inferir_tipos
from src.features import seleccionar_predictoras, preparar_matriz, construir_preprocesador
from src import arbol as ar, evaluate as ev

tipos_auditados = {'acc_now_delinq': 'Int8', 'acc_open_past_24mths': 'Int8', 'addr_state': 'category', 'all_util': 'Int16', 'annual_inc': 'float64', 'annual_inc_joint': 'float64', 'application_type': 'category', 'avg_cur_bal': 'Int32', 'bc_open_to_buy': 'Int32', 'bc_util': 'float64', 'chargeoff_within_12_mths': 'Int8', 'collection_recovery_fee': 'float64', 'collections_12_mths_ex_med': 'Int8', 'debt_settlement_flag': 'category', 'debt_settlement_flag_date': 'category', 'deferral_term': 'Int8', 'delinq_2yrs': 'Int8', 'delinq_amnt': 'Int32', 'desc': 'string', 'disbursement_method': 'category', 'dti': 'float64', 'dti_joint': 'float64', 'earliest_cr_line': 'string', 'emp_length': 'category', 'emp_title': 'string', 'fico_range_high': 'Int16', 'fico_range_low': 'Int16', 'funded_amnt': 'Int32', 'funded_amnt_inv': 'float64', 'grade': 'category', 'hardship_amount': 'float64', 'hardship_dpd': 'Int8', 'hardship_end_date': 'category', 'hardship_flag': 'category', 'hardship_last_payment_amount': 'float64', 'hardship_length': 'Int8', 'hardship_loan_status': 'category', 'hardship_payoff_balance_amount': 'float64', 'hardship_reason': 'category', 'hardship_start_date': 'category', 'hardship_status': 'category', 'hardship_type': 'category', 'home_ownership': 'category', 'id': 'string', 'il_util': 'Int16', 'initial_list_status': 'category', 'inq_fi': 'Int8', 'inq_last_12m': 'Int8', 'inq_last_6mths': 'Int8', 'installment': 'float64', 'int_rate': 'float64', 'issue_d': 'string', 'last_credit_pull_d': 'category', 'last_fico_range_high': 'Int16', 'last_fico_range_low': 'Int16', 'last_pymnt_amnt': 'float64', 'last_pymnt_d': 'category', 'loan_amnt': 'Int32', 'loan_status': 'category', 'max_bal_bc': 'Int32', 'member_id': 'string', 'mo_sin_old_il_acct': 'Int16', 'mo_sin_old_rev_tl_op': 'Int16', 'mo_sin_rcnt_rev_tl_op': 'Int16', 'mo_sin_rcnt_tl': 'Int16', 'mort_acc': 'Int8', 'mths_since_last_delinq': 'Int16', 'mths_since_last_major_derog': 'Int16', 'mths_since_last_record': 'Int16', 'mths_since_rcnt_il': 'Int16', 'mths_since_recent_bc': 'Int16', 'mths_since_recent_bc_dlq': 'Int16', 'mths_since_recent_inq': 'Int8', 'mths_since_recent_revol_delinq': 'Int16', 'next_pymnt_d': 'category', 'num_accts_ever_120_pd': 'Int8', 'num_actv_bc_tl': 'Int8', 'num_actv_rev_tl': 'Int8', 'num_bc_sats': 'Int8', 'num_bc_tl': 'Int8', 'num_il_tl': 'Int16', 'num_op_rev_tl': 'Int8', 'num_rev_accts': 'Int16', 'num_rev_tl_bal_gt_0': 'Int8', 'num_sats': 'Int8', 'num_tl_120dpd_2m': 'Int8', 'num_tl_30dpd': 'Int8', 'num_tl_90g_dpd_24m': 'Int8', 'num_tl_op_past_12m': 'Int8', 'open_acc': 'Int8', 'open_acc_6m': 'Int8', 'open_act_il': 'Int8', 'open_il_12m': 'Int8', 'open_il_24m': 'Int8', 'open_rv_12m': 'Int8', 'open_rv_24m': 'Int8', 'orig_projected_additional_accrued_interest': 'float64', 'out_prncp': 'float64', 'out_prncp_inv': 'float64', 'payment_plan_start_date': 'category', 'pct_tl_nvr_dlq': 'float64', 'percent_bc_gt_75': 'float64', 'policy_code': 'category', 'pub_rec': 'Int8', 'pub_rec_bankruptcies': 'Int8', 'purpose': 'category', 'pymnt_plan': 'category', 'recoveries': 'float64', 'revol_bal': 'Int32', 'revol_bal_joint': 'Int32', 'revol_util': 'float64', 'sec_app_chargeoff_within_12_mths': 'Int8', 'sec_app_collections_12_mths_ex_med': 'Int8', 'sec_app_earliest_cr_line': 'string', 'sec_app_fico_range_high': 'Int16', 'sec_app_fico_range_low': 'Int16', 'sec_app_inq_last_6mths': 'Int8', 'sec_app_mort_acc': 'Int8', 'sec_app_mths_since_last_major_derog': 'Int16', 'sec_app_num_rev_accts': 'Int8', 'sec_app_open_acc': 'Int8', 'sec_app_open_act_il': 'Int8', 'sec_app_revol_util': 'float64', 'settlement_amount': 'float64', 'settlement_date': 'category', 'settlement_percentage': 'float64', 'settlement_status': 'category', 'settlement_term': 'Int16', 'sub_grade': 'category', 'tax_liens': 'Int8', 'term': 'category', 'title': 'string', 'tot_coll_amt': 'Int32', 'tot_cur_bal': 'Int32', 'tot_hi_cred_lim': 'Int32', 'total_acc': 'Int16', 'total_bal_ex_mort': 'Int32', 'total_bal_il': 'Int32', 'total_bc_limit': 'Int32', 'total_cu_tl': 'Int8', 'total_il_high_credit_limit': 'Int32', 'total_pymnt': 'float64', 'total_pymnt_inv': 'float64', 'total_rec_int': 'float64', 'total_rec_late_fee': 'float64', 'total_rec_prncp': 'float64', 'total_rev_hi_lim': 'Int32', 'url': 'string', 'verification_status': 'category', 'verification_status_joint': 'category', 'zip_code': 'category'}
nombres_variables = {'acc_now_delinq': 'cuentas_en_mora_actual', 'acc_open_past_24mths': 'cuentas_abiertas_ultimos_24m', 'annual_inc': 'ingreso_anual', 'avg_cur_bal': 'saldo_promedio_cuentas', 'bc_open_to_buy': 'credito_disponible_tarjetas', 'bc_util': 'utilizacion_tarjetas_pct', 'chargeoff_within_12_mths': 'cuentas_castigadas_12m', 'collections_12_mths_ex_med': 'cobranzas_no_medicas_12m', 'delinq_2yrs': 'moras_ultimos_2_anios', 'delinq_amnt': 'monto_en_mora_actual', 'dti': 'ratio_deuda_ingreso', 'fico_range_high': 'fico_limite_superior', 'fico_range_low': 'fico_limite_inferior', 'inq_last_6mths': 'consultas_credito_6m', 'loan_amnt': 'monto_prestamo', 'mo_sin_old_il_acct': 'meses_desde_cuenta_plazos_mas_antigua', 'mo_sin_old_rev_tl_op': 'meses_desde_revolvente_mas_antigua', 'mo_sin_rcnt_rev_tl_op': 'meses_desde_revolvente_mas_reciente', 'mo_sin_rcnt_tl': 'meses_desde_cuenta_mas_reciente', 'mort_acc': 'cuentas_hipotecarias', 'mths_since_last_delinq': 'meses_desde_ultima_mora', 'mths_since_last_major_derog': 'meses_desde_ultima_mora_grave', 'mths_since_last_record': 'meses_desde_ultimo_registro_publico', 'mths_since_recent_bc': 'meses_desde_tarjeta_mas_reciente', 'mths_since_recent_bc_dlq': 'meses_desde_ultima_mora_tarjeta', 'mths_since_recent_inq': 'meses_desde_ultima_consulta_credito', 'mths_since_recent_revol_delinq': 'meses_desde_ultima_mora_revolvente', 'num_accts_ever_120_pd': 'cuentas_con_mora_120_dias', 'num_actv_bc_tl': 'tarjetas_bancarias_activas', 'num_actv_rev_tl': 'cuentas_revolventes_activas', 'num_bc_sats': 'tarjetas_bancarias_al_dia', 'num_bc_tl': 'total_cuentas_tarjeta', 'num_il_tl': 'total_cuentas_plazos', 'num_op_rev_tl': 'cuentas_revolventes_abiertas', 'num_rev_accts': 'total_cuentas_revolventes', 'num_rev_tl_bal_gt_0': 'revolventes_con_saldo', 'num_sats': 'cuentas_satisfactorias', 'num_tl_120dpd_2m': 'cuentas_mora_120_dias_ultimos_2m', 'num_tl_30dpd': 'cuentas_mora_30_dias_actual', 'num_tl_90g_dpd_24m': 'cuentas_mora_90_mas_dias_24m', 'num_tl_op_past_12m': 'cuentas_abiertas_12m', 'open_acc': 'lineas_credito_abiertas', 'pct_tl_nvr_dlq': 'pct_cuentas_sin_mora', 'percent_bc_gt_75': 'pct_tarjetas_utilizacion_mayor_75', 'pub_rec': 'registros_publicos_adversos', 'pub_rec_bankruptcies': 'quiebras_registradas', 'revol_bal': 'saldo_credito_revolvente', 'revol_util': 'utilizacion_credito_revolvente_pct', 'tax_liens': 'gravamenes_fiscales', 'term': 'plazo_meses', 'tot_coll_amt': 'monto_total_cobranzas', 'tot_cur_bal': 'saldo_total_actual', 'tot_hi_cred_lim': 'limite_credito_total', 'total_acc': 'total_lineas_credito', 'total_bal_ex_mort': 'saldo_total_sin_hipoteca', 'total_bc_limit': 'limite_total_tarjetas', 'total_il_high_credit_limit': 'limite_total_creditos_plazos', 'total_rev_hi_lim': 'limite_total_credito_revolvente', 'emp_length': 'antiguedad_laboral', 'home_ownership': 'tenencia_vivienda', 'verification_status': 'verificacion_ingresos', 'purpose': 'proposito_prestamo', 'addr_state': 'estado_residencia', 'application_type': 'tipo_solicitud', 'earliest_cr_line': 'fecha_primera_linea_credito'}
variables_rfe30 = ['loan_amnt', 'term', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'total_acc', 'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'mo_sin_old_rev_tl_op', 'mths_since_recent_bc', 'num_bc_sats', 'num_rev_tl_bal_gt_0', 'num_tl_op_past_12m', 'percent_bc_gt_75', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit', 'total_il_high_credit_limit', 'meses_desde_earliest_cr_line']
mapa_derivadas = {"meses_desde_earliest_cr_line": "earliest_cr_line"}
variables_rfe30_fuente = [mapa_derivadas.get(c,c) for c in variables_rfe30]
assert len(variables_rfe30)==30 and len(set(variables_rfe30))==30
archivo=localizar_datos()
print("Dataset:",archivo)
print("Variables seleccionadas:",len(variables_rfe30))

Dataset: C:\Users\josue\Downloads\accepted_2007_to_2018Q4.csv.gz
Variables seleccionadas: 30


## 1. Leer datos y mantener la partición temporal

Usamos los tipos conservadores auditados y las mismas columnas del escenario base. Solo los préstamos con `Fully Paid` o `Charged Off` participan. Se aparta prueba y no se utiliza en el ajuste ni en la comparación.

In [2]:
columnas_fuente=pd.read_csv(archivo,nrows=0).columns.tolist()
tipos_inferidos,_=inferir_tipos(archivo)
tabla_columnas=pd.DataFrame(columns=columnas_fuente)
candidatas=seleccionar_predictoras(tabla_columnas,[],incluir_precio_lc=False)
columnas_lectura=[c for c in columnas_fuente if c in candidatas+["loan_status","issue_d"]]
dtype_lectura={}
for c in columnas_lectura:
    tipo=tipos_auditados.get(c,tipos_inferidos[c])
    dtype_lectura[c]=tipo if tipo not in ["category","string"] else "string"
inicio=time.perf_counter()
df=pd.read_csv(archivo,usecols=columnas_lectura,dtype=dtype_lectura,skipinitialspace=True,low_memory=False)
for c in df.select_dtypes(include="string").columns: df[c]=df[c].astype("category")
for c in ["issue_d","earliest_cr_line","sec_app_earliest_cr_line"]:
    if c in df.columns: df[c]=pd.to_datetime(df[c].astype("string"),format="%b-%Y",errors="coerce")
if "term" in df.columns:
    df["term"]=pd.to_numeric(df["term"].astype("string").str.strip().str.replace(" months","",regex=False),errors="coerce").astype("Int8")
columnas_vacias=df.columns[df.isna().all()].tolist()
if columnas_vacias:
    df=df.drop(columns=columnas_vacias); candidatas=[c for c in candidatas if c not in columnas_vacias]
df=df.loc[df.loan_status.isin(["Fully Paid","Charged Off"])].copy()
df["default"]=df.loan_status.map({"Fully Paid":0,"Charged Off":1}).astype("int8")
df["fecha_prestamo"]=df.issue_d
df["anio"]=df.issue_d.dt.year
assert df.anio.notna().all()
train_df=df.loc[df.anio<ANIO_CORTE_VALIDACION].copy()
valid_df=df.loc[df.anio.between(ANIO_CORTE_VALIDACION,ANIO_CORTE_PRUEBA-1)].copy()
prueba_n=int((df.anio>=ANIO_CORTE_PRUEBA).sum())
print(f"Lectura y preparación: {time.perf_counter()-inicio:.1f} s")
print(f"Entrenamiento: {len(train_df):,}; validación: {len(valid_df):,}; prueba reservada: {prueba_n:,}")
print("Tasa Charged Off — entrenamiento:",round(train_df.default.mean(),4),"validación:",round(valid_df.default.mean(),4))

Lectura y preparación: 48.5 s
Entrenamiento: 451,059; validación: 668,640; prueba reservada: 225,611
Tasa Charged Off — entrenamiento: 0.1696 validación: 0.2154


## 2. Reconstruir las 65 variables y seleccionar las 30

Replicamos el filtro por faltantes del notebook final, calculado solo con entrenamiento. Las 30 seleccionadas son variables originales; la fecha de la línea de crédito más antigua se lee desde `earliest_cr_line` y se transforma en meses de antigüedad.

In [3]:
faltantes_train=(train_df[candidatas].isna().mean()*100)
columnas_mas95=faltantes_train[faltantes_train>UMBRAL_FALTANTES].index.tolist()
train_raw=preparar_matriz(train_df[candidatas+["fecha_prestamo"]]).drop(columns="fecha_prestamo")
valid_raw=preparar_matriz(valid_df[candidatas+["fecha_prestamo"]]).drop(columns="fecha_prestamo")
mapa_fecha={"earliest_cr_line":"meses_desde_earliest_cr_line","sec_app_earliest_cr_line":"meses_desde_sec_app_earliest_cr_line"}
excluidas=[mapa_fecha.get(c,c) for c in columnas_mas95]
variables65=[c for c in train_raw.columns if c not in excluidas]
assert len(variables65)==65, f"Esperadas 65; encontradas {len(variables65)}"
assert all(c in variables65 for c in variables_rfe30), "La selección guardada no coincide con el EDA final."
variables30_fuente=[c for c in variables_rfe30_fuente if c in candidatas]
assert len(variables30_fuente)==30
print("Predictoras base después del filtro de faltantes:",len(variables65))
print("Predictoras del RFE disponibles en el archivo:",len(variables30_fuente))
display(pd.DataFrame({"nombre_original":variables_rfe30,
                      "nombre_espanol":[nombres_variables.get(c,c) for c in variables_rfe30]}))

Predictoras base después del filtro de faltantes: 65
Predictoras del RFE disponibles en el archivo: 30


,nombre_original,nombre_espanol
0,loan_amnt,monto_prestamo
1,term,plazo_meses
2,emp_length,antiguedad_laboral
3,home_ownership,tenencia_vivienda
4,annual_inc,ingreso_anual
5,verification_status,verificacion_ingresos
6,purpose,proposito_prestamo
7,addr_state,estado_residencia
8,dti,ratio_deuda_ingreso
9,delinq_2yrs,moras_ultimos_2_anios


## 3. Entrenar regresión logística con 65 y con 30 variables

La regresión logística se prepara como en el baseline: mediana para numéricas, categoría `Sin información`, One-Hot Encoding y escalado. Ambas logísticas usan exactamente los mismos préstamos y la misma validación.

In [4]:
def preparar_logistica(columnas):
    xt=train_raw[columnas]
    xv=valid_raw[columnas]
    prep,_=construir_preprocesador(xt,con_indicador=[])
    matriz_train=prep.fit_transform(xt)
    matriz_valid=prep.transform(xv)
    return matriz_train,matriz_valid,prep

y_train=train_df.default.to_numpy(); y_valid=valid_df.default.to_numpy()
X65_train,X65_valid,prep65=preparar_logistica(variables65)
X30_train,X30_valid,prep30=preparar_logistica(variables_rfe30)
logistica65=LogisticRegression(C=1.0,max_iter=2000,solver="lbfgs",random_state=SEMILLA).fit(X65_train,y_train)
logistica30=LogisticRegression(C=1.0,max_iter=2000,solver="lbfgs",random_state=SEMILLA).fit(X30_train,y_train)
score_log65=logistica65.predict_proba(X65_valid)[:,1]
score_log30=logistica30.predict_proba(X30_valid)[:,1]
print("Columnas codificadas — logística 65:",X65_train.shape[1])
print("Columnas codificadas — logística 30:",X30_train.shape[1])

C:\Users\josue\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [4, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


C:\Users\josue\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Columnas codificadas — logística 65: 139
Columnas codificadas — logística 30: 105


## 4. Ajustar y comparar el árbol de decisión con las 30 variables

El árbol no requiere escalar numéricas. Las categorías se rellenan y codifican, agrupando niveles que aparecen en menos de 0.5% del entrenamiento. Elegimos profundidad, tamaño mínimo de hoja y peso de clases por ROC-AUC en validación, como en el notebook de árbol, pero la búsqueda se reduce a 30 combinaciones para que pueda ejecutarse sobre toda la data.

In [5]:
# Para el árbol usamos las 30 columnas ya preparadas, incluida la antigüedad derivada.
# Las variables numéricas se imputan con medianas de entrenamiento para que el
# árbol reciba valores compatibles y el tratamiento sea igual al baseline.
tree_train=train_raw[variables_rfe30].copy()
tree_valid=valid_raw[variables_rfe30].copy()
numericas_tree=tree_train.select_dtypes(include="number").columns.tolist()
medianas_tree=tree_train[numericas_tree].astype("float64").median()
for c in numericas_tree:
    tree_train[c]=pd.to_numeric(tree_train[c],errors="coerce").astype("float64").fillna(medianas_tree[c])
    tree_valid[c]=pd.to_numeric(tree_valid[c],errors="coerce").astype("float64").fillna(medianas_tree[c])
assert tree_train[numericas_tree].isna().sum().sum()==0

auxiliares_tree=["loan_status","default","fecha_prestamo","anio"]
particiones_tree={
    "entrenamiento":pd.concat([tree_train,train_df[auxiliares_tree]],axis=1),
    "validacion":pd.concat([tree_valid,valid_df[auxiliares_tree]],axis=1),
}
datos_arbol=ar.preparar_datos(particiones_tree,columnas_vacias,incluir_precio_lc=False)
print("Predictoras del árbol:",datos_arbol["variables"]["predictoras_finales"])
print("Columnas tras codificar:",datos_arbol["variables"]["columnas_modelo"])
print("Matrices:",datos_arbol["X"]["entrenamiento"].shape,datos_arbol["X"]["validacion"].shape)

rejilla={"max_depth":[3,5,8,10,12],
         "min_samples_leaf":[500,1000,2000],
         "class_weight":[None,"balanced"]}
busqueda=ar.buscar_hiperparametros(datos_arbol,grilla=rejilla,verbose=True)
mejor=busqueda.iloc[0]
parametros=ar.parametros_de_fila(mejor)
print("Parámetros seleccionados:",parametros)
print("ROC-AUC train/validación:",round(mejor.auc_entrenamiento,4),round(mejor.auc_validacion,4))
print("Profundidad real y hojas:",int(mejor.profundidad_real),int(mejor.hojas))

Predictoras del árbol: 30
Columnas tras codificar: 93
Matrices: (451059, 93) (668640, 93)


  20/30 combinaciones (92s)


Parámetros seleccionados: {'max_depth': 12, 'min_samples_leaf': 1000, 'class_weight': 'balanced'}
ROC-AUC train/validación: 0.6938 0.6878
Profundidad real y hojas: 12 326


## 5. Resultados

Comparamos ROC-AUC y PR-AUC, que no dependen de elegir un umbral, y añadimos recall/F1 con umbral seleccionado en validación. La fila principal para responder la pregunta es `Árbol con 30` frente a `Logística baseline con 65`; la logística con 30 permite ver si el cambio viene de reducir variables o del tipo de modelo.

In [6]:
arbol30,scores_arbol,resultados_arbol=ar.entrenar_y_evaluar(datos_arbol,parametros)
umbral65=ev.elegir_umbral(y_valid,score_log65)
umbral30=ev.elegir_umbral(y_valid,score_log30)
resultados_validacion={
    "Logística baseline con 65":ev.evaluar(y_valid,score_log65,umbral65),
    "Logística con RFE 30":ev.evaluar(y_valid,score_log30,umbral30),
    "Árbol con RFE 30":resultados_arbol["validacion"],
}
tabla=pd.DataFrame([{
    "modelo":nombre,
    "variables_originales":65 if nombre.endswith("65") else 30,
    "ROC-AUC":metrica["roc_auc"],
    "PR-AUC":metrica["pr_auc"],
    "Recall":metrica["recall"],
    "Precision":metrica["precision"],
    "F1":metrica["f1"],
    "umbral":metrica["umbral"],
} for nombre,metrica in resultados_validacion.items()])
display(tabla.round(4))
print("Diferencia de ROC-AUC árbol 30 vs baseline logístico 65:",
      round(resultados_validacion["Árbol con RFE 30"]["roc_auc"]-resultados_validacion["Logística baseline con 65"]["roc_auc"],4))
print("Prueba reservada no se evaluó.")

,modelo,variables_originales,ROC-AUC,PR-AUC,Recall,Precision,F1,umbral
0,Logística baseline con 65,65,0.7156,0.4093,0.6265,0.3530,0.4515,0.1800
1,Logística con RFE 30,30,0.7149,0.4086,0.6387,0.3485,0.4509,0.1778
2,Árbol con RFE 30,30,0.6878,0.3763,0.6694,0.3135,0.4271,0.4838


Diferencia de ROC-AUC árbol 30 vs baseline logístico 65: -0.0278
Prueba reservada no se evaluó.


## 6. Conclusión

La respuesta se basa en ROC-AUC de validación; PR-AUC, recall y F1 ayudan a revisar la detección de la clase minoritaria. El árbol se eligió con la misma validación usada para medirlo, por lo que estos resultados sirven para comparar candidatos, no como evaluación final independiente. La prueba permanece intacta.